In [ ]:
import os
import glob
import pickle
import pandas as pd
import numpy as np
import networkx as nx

from dask.diagnostics import ProgressBar
import scipy.sparse
from scipy.stats import wilcoxon
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import fdrcorrection
from scipy.stats import pearsonr
from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2
from dask.distributed import Client, LocalCluster

from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase
from pyscenic.utils import modules_from_adjacencies, load_motifs
from pyscenic.prune import prune2df, df2regulons
from pyscenic.aucell import aucell

import seaborn as sns
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

from pyscenic.cli.utils import load_exp_matrix
from pyscenic.prune import prune2df, df2regulons
from pyscenic.export import export2loom
from pyscenic.aucell import aucell
#from pyscenic.rnkdb import FeatherRankingDatabase as RankingDatabase
from pyscenic.utils import modules_from_adjacencies
from dask.diagnostics import ProgressBar

In [ ]:
## 导入数据库

In [ ]:
DATABASE_FOLDER = 'data/external/cistarget_databases/'
OUTPUT_DIR = "results/pyscenic/pyscenic_output_file"

adata_path = 'data/processed/sc_reference_file'
DATABASES_GLOB = os.path.join(DATABASE_FOLDER, "*rankings.feather")
# 需要从 httpdata/local_or_external_file 下载

MOTIF_ANNOTATIONS_FNAME = os.path.join(DATABASE_FOLDER, "motifs-v9-nr.hgnc-m0.001-o0.0.tbl.txt")
# httpdata/local_or_external_file

REGULONS_FNAME = os.path.join(OUTPUT_DIR, "regulons.p")
MOTIFS_FNAME = os.path.join(OUTPUT_DIR, "motifs.csv")

In [ ]:
# 加载数据
adata = sc.read_h5ad(adata_path)
#sc.pp.subsample(adata, fraction=0.1)
adata.layers['counts'] = adata.X.copy()

# 准备表达矩阵
ex_matrix = pd.DataFrame(
    adata.layers['counts'].toarray() if scipy.sparse.issparse(adata.layers['counts']) else adata.layers['counts'],
    columns=adata.var_names,
    index=adata.obs_names
)

# 加载TF列表
tf_list_path = "results/pyscenic/pyscenic_output_file"
tf_names = load_tf_names(tf_list_path)

In [ ]:
client = Client(n_workers=60, threads_per_worker=4)
# 运行 GRNBoost2
adjacencies = grnboost2(
    expression_data=ex_matrix,
    tf_names=tf_names,
    client_or_address=client,
    verbose=True# 显式传递 client
)

In [ ]:
# 保存结果
adjacencies.head()
adjacencies.to_csv(os.path.join(OUTPUT_DIR, "EXN_nasa_adjacencies.csv"), index=False)
print(f"生成的调控关系数量: {len(adjacencies)}")

In [ ]:
db_fnames = glob.glob(DATABASES_GLOB)
def name(fname):
    return os.path.splitext(os.path.basename(fname))[0]
dbs = [RankingDatabase(fname=fname, name=name(fname)) for fname in db_fnames]
dbs

In [ ]:
#从共表达网络构建模块
modules = list(modules_from_adjacencies(adjacencies, ex_matrix))

# 提取所有 Regulon 的 TF、靶基因和权重
data = []
for regulon in modules:
    tf = regulon.name.replace('Regulon for ', '')  # 从 "Regulon for HMGA1" 提取 "HMGA1"
    for target, weight in regulon.gene2weight.items():
        data.append({"TF": tf, "Target": target, "Weight": weight})

# 转换为 DataFrame 并保存
df = pd.DataFrame(data)
df.to_csv(os.path.join(OUTPUT_DIR, "EXN_nasa_regulons_tf_target_weight.csv"), index=False)

In [ ]:
# 读取 CSV 文件
df = pd.read_csv(os.path.join(OUTPUT_DIR, "EXN_nasa_regulons_tf_target_weight.csv") ) # 替换为你的文件路径

# 按 TF 分组，统计每个 TF 的靶基因数量
tf_stats = df.groupby("TF").agg(
    Num_Targets=("Target", "count"),      # 统计靶基因数量
    Target_List=("Target", lambda x: "、".join(x)),  # 用顿号连接所有靶基因
    Mean_Weight=("Weight", "mean")        # 计算平均权重（可选）
).reset_index()

# 保存结果
tf_stats.to_csv(os.path.join(OUTPUT_DIR, "EXN_nasa_summary.csv"), index=False, encoding="utf-8-sig")
print("统计结果已保存为: EXN_nasa_summary.csv")

In [ ]:
# 计算富集基序
# Calculate a list of enriched motifs and the corresponding target genes for all modules.
with ProgressBar():
    df = prune2df(dbs, modules, MOTIF_ANNOTATIONS_FNAME)

# Create regulons from this table of enriched motifs.
regulons = df2regulons(df)

In [ ]:
# Save the enriched motifs and the discovered regulons to disk.
df.to_csv(MOTIFS_FNAME)
with open(REGULONS_FNAME, "wb") as f:
    pickle.dump(regulons, f)

num_regulons = len(regulons)
print(f"Number of regulons: {num_regulons}")

In [ ]:
## 可视化

In [ ]:
OUTPUT_DIR = "results/pyscenic/pyscenic_output_file"
adata_path = 'data/processed/sc_reference_file'

REGULONS_FNAME = 'results/pyscenic/pyscenic_output_file'
MOTIFS_FNAME = 'results/pyscenic/pyscenic_output_file'

In [ ]:
regulons_path = os.path.join(OUTPUT_DIR, "regulons.p")
with open(regulons_path, "rb") as f:
    regulons = pickle.load(f)

In [ ]:
adata = sc.read_h5ad(adata_path)
adata.layers['counts'] = adata.X.copy()
print(f"表达矩阵形状: {adata.layers['counts'].shape}") 

matrix = pd.DataFrame(
    adata.layers['counts'].toarray() if scipy.sparse.issparse(adata.layers['counts']) else adata.layers['counts'],
    columns=adata.var_names,
    index=adata.obs_names
)

In [ ]:
matrix = matrix.loc[adata.obs.index, :]
matrix = matrix.loc[:, adata.var.index]

In [ ]:
auc_mtx = pd.read_csv(os.path.join(OUTPUT_DIR, "EXN_nasa_auc_matrix.csv"),index_col=0)

In [ ]:
annotations = adata.obs["annotation"]

# 对齐注释与当前auc_mtx的细胞
filtered_annotations = annotations[annotations.index.isin(auc_mtx.index)]

# 去除注释中的“Other”标签细胞
filtered_annotations = filtered_annotations[filtered_annotations != "Other"]

# 按细胞类型分组计算平均AUC（保留分组顺序）
grouped_auc_mtx = (
    auc_mtx
    .assign(cell_type=filtered_annotations)  # 添加细胞类型列
    .groupby('cell_type', observed=True)
    .mean()
    .reindex(annotations.cat.categories)  # 按原始类别顺序排序
    .dropna()  # 移除空分组
)

In [ ]:
## 富集因子分组热图

In [ ]:
g = sns.clustermap(grouped_auc_mtx.T, col_cluster=True, figsize=(12, 10),
                   col_colors=None,  # 不需要行颜色条
                   row_cluster=False)  # 不对行进行聚类

g.ax_heatmap.set_xticklabels(grouped_auc_mtx.index, rotation=90)
g.savefig(os.path.join(OUTPUT_DIR, "EXN_nasa_motif_heatmap_grouped_90.pdf"), format="pdf", bbox_inches="tight")

In [ ]:
## 绘制共表达网络图

In [ ]:
df = pd.read_csv("results/pyscenic/pyscenic_output_file")

# 筛选涉及 BDNF 和 ADCYAP1 的调控关系
genes_of_interest = ['BDNF', 'ADCYAP1']
filtered_df = df[(df['Target'].isin(genes_of_interest)) | (df['TF'].isin(genes_of_interest))]
# 继续筛选 Weight 大于 0.5 的行
filtered_df_high_weight = filtered_df[filtered_df['Weight'] > 0.5]
filtered_df_high_weight

In [ ]:
G = nx.DiGraph()

# 添加节点和边
for index, row in filtered_df_high_weight.iterrows():
    tf = row['TF']
    target = row['Target']
    weight = row['Weight']
    G.add_edge(tf, target, weight=weight)

# 自定义布局
pos = {}

# 将 BDNF 和其相关节点放在左边
bdnf_nodes = [node for node in G.nodes if node == 'BDNF' or 'BDNF' in [edge[1] for edge in G.in_edges(node)]]
for i, node in enumerate(bdnf_nodes):
    pos[node] = (-1, i * 1.5)

# 将 ADCYAP1 和其相关节点放在右边
adcyap1_nodes = [node for node in G.nodes if node == 'ADCYAP1' or 'ADCYAP1' in [edge[1] for edge in G.in_edges(node)]]
for i, node in enumerate(adcyap1_nodes):
    pos[node] = (1, i * 1.5)

# 检查是否有节点没有被分配位置，并为这些节点分配默认位置
remaining_nodes = set(G.nodes) - set(pos.keys())
if remaining_nodes:
    # 使用 spring_layout 为剩余节点分配位置
    spring_pos = nx.spring_layout(G.subgraph(remaining_nodes), scale=2)
    pos.update(spring_pos)

# 绘制网络图
plt.figure(figsize=(12, 8))
# 增大节点的大小
nx.draw_networkx_nodes(G, pos, node_size=5000, node_color='skyblue', edgecolors='black', linewidths=1)
# 根据权重调整边的宽度
edge_widths = [G[u][v]['weight'] * 2 for u, v in G.edges()]
nx.draw_networkx_edges(G, pos, arrowstyle='-|>', arrowsize=20, edge_color='gray', width=edge_widths)
# 增大字体大小
nx.draw_networkx_labels(G, pos, font_size=15)
#plt.title('Regulatory Network for BDNF and ADCYAP1', fontsize=18, fontweight='bold')

# 显示图形
plt.axis('off')  # 关闭坐标轴
plt.tight_layout()  # 自动调整子图参数，使之填充整个图像区域
plt.savefig("results/pyscenic/pyscenic_output_file", format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
## 保存绘图基因列表

In [ ]:
df = pd.read_csv("results/pyscenic/pyscenic_output_file")

# 筛选涉及 BDNF 和 ADCYAP1 的调控关系
genes_of_interest = ['BDNF', 'ADCYAP1']
filtered_df = df[(df['Target'].isin(genes_of_interest)) & (df['Weight'] > 0.5)]

# 提取与 BDNF 和 ADCYAP1 互作的基因列表
bdnf_interacting_genes = filtered_df[filtered_df['Target'] == 'BDNF']['TF'].tolist()
adcyap1_interacting_genes = filtered_df[filtered_df['Target'] == 'ADCYAP1']['TF'].tolist()

# 创建一个新的 DataFrame
interactions_df = pd.DataFrame({
    'Gene': bdnf_interacting_genes + adcyap1_interacting_genes,
    'Interacts_with_BDNF': [gene if gene in bdnf_interacting_genes else None for gene in bdnf_interacting_genes + adcyap1_interacting_genes],
    'Interacts_with_ADCYAP1': [gene if gene in adcyap1_interacting_genes else None for gene in bdnf_interacting_genes + adcyap1_interacting_genes]
})

# 保存到本地 CSV 文件
interactions_df.to_csv("results/pyscenic/pyscenic_output_file", index=False)

In [ ]:
# 读取 CSV 文件
df = pd.read_csv("results/pyscenic/pyscenic_output_file")

# 筛选涉及 BDNF 和 ADCYAP1 的调控关系
genes_of_interest = ['BDNF', 'ADCYAP1']
filtered_df = df[(df['Target'].isin(genes_of_interest)) & (df['Weight'] > 0.5)]

# 提取特定基因列表
specific_genes = ['ATF4', 'ADCYAP1', 'BHLHE40', 'CUX1', 'CUX2', 'EGR1', 'BDNF', 'EMX1', 'ETV3', 'FOS', 'FOSL2', 'GADD45A', 'HES4', 'HSF4', 'HSPA5', 'ID2', 'JUNB', 'KDM4B', 'NR4A1', 'NR4A2', 'PKM', 'SEMA4A', 'UBB', 'ZBTB16', 'ZNF436', 'NUAK2']

# 创建两个新的 DataFrame 来存储结果
bdnf_targets_df = pd.DataFrame(columns=['TF', 'Target'])
adcyap1_targets_df = pd.DataFrame(columns=['TF', 'Target'])

# 遍历特定基因列表，查找它们在 filtered_df_high_weight 中的分类
for gene in specific_genes:
    bdnf_target_rows = filtered_df[(filtered_df['TF'] == gene) & (filtered_df['Target'] == 'BDNF')]
    adcyap1_target_rows = filtered_df[(filtered_df['TF'] == gene) & (filtered_df['Target'] == 'ADCYAP1')]
    
    if not bdnf_target_rows.empty:
        for _, row in bdnf_target_rows.iterrows():
            if not bdnf_targets_df[(bdnf_targets_df['TF'] == row['TF']) & (bdnf_targets_df['Target'] == row['Target'])].empty:
                continue
            bdnf_targets_df = bdnf_targets_df.append(row[['TF', 'Target']], ignore_index=True)
    
    if not adcyap1_target_rows.empty:
        for _, row in adcyap1_target_rows.iterrows():
            if not adcyap1_targets_df[(adcyap1_targets_df['TF'] == row['TF']) & (adcyap1_targets_df['Target'] == row['Target'])].empty:
                continue
            adcyap1_targets_df = adcyap1_targets_df.append(row[['TF', 'Target']], ignore_index=True)

# 保存到本地 CSV 文件
bdnf_targets_df.to_csv("results/pyscenic/pyscenic_output_file", index=False)
adcyap1_targets_df.to_csv("results/pyscenic/pyscenic_output_file", index=False)